In [20]:
from ase.io.vasp import read_vasp_xdatcar, write_vasp
from ase import Atoms
import numpy as np
from ase.neb import NEB


In [2]:
def interpolate_configs(config1: Atoms, config2: Atoms, num_interpolations: int = 10):
    """
    Интерполяция конфигураций между двумя заданными конфигурациями, с учетом разного количества атомов.
    """
    interpolated_configs = []

    num_atoms1 = len(config1)
    num_atoms2 = len(config2)

    if num_atoms1 != num_atoms2:
        # Находим минимальное количество атомов и используем его для интерполяции
        num_atoms = min(num_atoms1, num_atoms2)
        config1 = config1[:num_atoms]
        config2 = config2[:num_atoms]

    for i in range(1, num_interpolations + 1):
        alpha = i / (num_interpolations + 1)
        positions = config1.get_positions() * (1 - alpha) + config2.get_positions() * alpha
        new_config = config1.copy()
        new_config.set_positions(positions)
        interpolated_configs.append(new_config)

    return interpolated_configs

def merge_xdatcar(file1, file2, time_steps1, time_steps2, output_file):
    # Чтение файлов XDATCAR
    configs1 = read_vasp_xdatcar(file1, index=slice(None))
    configs2 = read_vasp_xdatcar(file2, index=slice(None))

    merged_configs = []

    i, j = 0, 0

    while i < len(time_steps1) and j < len(time_steps2):
        time_step1 = time_steps1[i]
        time_step2 = time_steps2[j]

        if abs(time_step1 - time_step2) < 1e-3:  # Считаем временные шаги совпадающими, если их разница очень мала
            # Добавляем конфигурацию с минимальным числом атомов
            if len(configs1[i]) <= len(configs2[j]):
                merged_configs.append(configs1[i])
            else:
                merged_configs.append(configs2[j])

            # Интерполяция между двумя конфигурациями
            interpolated = interpolate_configs(configs1[i], configs2[j])
            merged_configs.extend(interpolated)

            i += 1
            j += 1
        elif time_step1 < time_step2:
            i += 1
        else:
            j += 1

    # Запись конфигураций в новый файл XDATCAR
    with open(output_file, 'w') as f:
        # Запись первой конфигурации
        write_vasp(f, merged_configs[0], direct=True)

    # Запись всех остальных конфигураций
    for config in merged_configs[1:]:
        with open(output_file, 'a') as f:
            f.write('\n')  # Разделяем каждую запись новой строкой
            write_vasp(f, config, direct=True)


# Пример массивов времени (замените их своими данными)
time_steps1 = np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1]
time_steps2 = np.loadtxt("out_O.txt", usecols=range(6, 10)).T[1]

# Использование функции
merge_xdatcar('XDATCAR_Fe', 'XDATCAR_O', time_steps1, time_steps2, 'XDATCAR_merged.txt')


In [94]:
time_steps1 = np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1]
time_steps2 = np.loadtxt("out_O.txt", usecols=range(6, 10)).T[1]
configs1 = read_vasp_xdatcar('XDATCAR_Fe', index=slice(None))
configs2 = read_vasp_xdatcar('XDATCAR_O', index=slice(None))
file = open('XDATCAR', 'w')
file.write('Comment\n')
file.write('1.0\n')
formatted_matrix = "\n".join("{:.6f} {:.6f} {:.6f}".format(*row) for row in configs1[0].get_cell())
file.write(formatted_matrix)
file.write('\n')
file.write(configs1[0].get_chemical_symbols()[0] + ' ' + configs2[0].get_chemical_symbols()[0])
file.write('\n')
file.write(str(configs1[0].get_number_of_atoms()) + ' ' + str(configs2[0].get_number_of_atoms()))

num_atoms = configs1[0].get_number_of_atoms() + configs2[0].get_number_of_atoms()

file.write('\n')
i=1
file.write('Direct configuration=     {:.6f}'.format(i))
file.write('\n')


file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs1[0].get_positions()) )
file.write('\n')
file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs2[0].get_positions()) )

file.write('\n')

i = 1
j = 0
count = 1


while i < (10*len(time_steps1)+len(time_steps1)):
        
    if abs(time_steps1[count] - time_steps2[count]) < 0.05 :
        j = j + 1
        
    #interpolated_1 = interpolate_configs(configs1[count-1], configs1[count])
    #interpolated_2 = interpolate_configs(configs2[count-1], configs2[count])

    interpolated_1 = [configs1[count-1]] + [configs1[count-1].copy() for _ in range(10)] + [configs1[count]]
    neb = NEB(interpolated_1)
    neb.interpolate()

    interpolated_2 = [configs2[count-1]] + [configs2[count-1].copy() for _ in range(10)] + [configs2[count]]
    neb = NEB(interpolated_2)
    neb.interpolate()

    
    for k in range(0, len(interpolated_1)):
        
        file.write('Direct configuration=     {:.6f}\n'.format(i+1))
        file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_1[k].get_positions()) )
        file.write('\n')
        file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_2[k].get_positions()) )
        file.write('\n')
        
        i = i + 1
        
    
    file.write('Direct configuration=     {:.6f}\n'.format(i+1))
    file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs1[count].get_positions()) )
    file.write('\n')
    file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs2[j].get_positions()) )
    file.write('\n')
    
    i = i + 1
    count = count + 1
    

file.close()

In [73]:
import numpy as np
from ase.io.vasp import read_vasp_xdatcar, write_vasp
from ase.neb import NEB

# Загрузка временных шагов и конфигураций
time_steps1 = np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1]
time_steps2 = np.loadtxt("out_O.txt", usecols=range(6, 10)).T[1]
configs1 = read_vasp_xdatcar('XDATCAR_Fe', index=slice(None))
configs2 = read_vasp_xdatcar('XDATCAR_O', index=slice(None))

# Открываем файл для записи
file = open('XDATCAR', 'w')
file.write('Comment\n')
file.write('1.0\n')
formatted_matrix = "\n".join("{:.6f} {:.6f} {:.6f}".format(*row) for row in configs1[0].get_cell())
file.write(formatted_matrix + '\n')

# Запись информации о типах и количестве атомов
file.write(f"{configs1[0].get_chemical_symbols()[0]} {configs2[0].get_chemical_symbols()[0]}\n")
file.write(f"{configs1[0].get_number_of_atoms()} {configs2[0].get_number_of_atoms()}\n")

i, j, count = 1, 1, 1

# Основной цикл по временным шагам
while count < len(time_steps1):
    if abs(time_steps1[count] - time_steps2[j-1]) < 1:
        print("yes")
        j = j + 1
        # Создание интерполированных конфигураций
        interpolated_1 = [configs1[count - 1]] + [configs1[count - 1].copy() for _ in range(10)] + [configs1[count]]
        neb1 = NEB(interpolated_1)
        neb1.interpolate(method='idpp')
        for y in range(0, 12):
            interpolated_1[y].wrap()
        

        interpolated_2 = [configs2[count - 1]] + [configs2[count - 1].copy() for _ in range(10)] + [configs2[count]]
        neb2 = NEB(interpolated_2)
        neb2.interpolate(method='idpp')
        for y in range(0, 12):
            interpolated_2[y].wrap()

        for k in range(len(interpolated_1)):
            file.write(f'Direct configuration=     {i:.6f}\n')
            file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_1[k].get_positions()))
            file.write('\n')
            file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_2[k].get_positions()))
            file.write('\n')
            i += 1

    # Запись последней конфигурации
    file.write(f'Direct configuration=     {i:.6f}\n')
    file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs1[count].get_positions()))
    file.write('\n')
    file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs2[count].get_positions()))
    file.write('\n')

    i += 1
    count += 1

file.close()


/Users/oq7884/opt/anaconda3/lib/python3.9/site-packages/ase/atoms.py:966: VisibleDeprecationWarning: Use get_global_number_of_atoms() instead
  warnings.warn('Use get_global_number_of_atoms() instead',


yes
yes
yes
yes
yes
yes


KeyboardInterrupt: 

In [87]:
configs1[1].wrap()

In [88]:
configs1 = read_vasp_xdatcar('XDATCAR_Fe', index=slice(None))

In [93]:
(configs1[1].get_positions() - configs1[0].get_positions())

array([[ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [15.0603013 ,  0.        ,  2.15147161],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.   

In [112]:
import numpy as np
from ase.io.vasp import read_vasp_xdatcar
from ase.neb import NEB

# Загрузка временных шагов и конфигураций
time_steps1 = np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1]
time_steps2 = np.loadtxt("out_O.txt", usecols=range(6, 10)).T[1]
configs1 = read_vasp_xdatcar('XDATCAR_Fe', index=slice(None))
configs2 = read_vasp_xdatcar('XDATCAR_O', index=slice(None))

# Установка периодических граничных условий для всех конфигураций
for config in configs1 + configs2:
    config.set_pbc(True)

# Открываем файл для записи
file = open('XDATCAR', 'w')
file.write('Comment\n')
file.write('1.0\n')
formatted_matrix = "\n".join("{:.6f} {:.6f} {:.6f}".format(*row) for row in configs1[0].get_cell())
file.write(formatted_matrix + '\n')

# Запись информации о типах и количестве атомов
file.write(f"{configs1[0].get_chemical_symbols()[0]} {configs2[0].get_chemical_symbols()[0]}\n")
file.write(f"{configs1[0].get_number_of_atoms()} {configs2[0].get_number_of_atoms()}\n")

i, j, count = 1, 1, 1

# Основной цикл по временным шагам
while count < len(time_steps1) and j < len(time_steps2):
    if time_steps1[count] > time_steps2[j] :
        # Временные шаги совпадают, выполняем интерполяцию для обеих конфигураций
        j += 1

        # Получение скалированных позиций для интерполяции
        scaled_pos1_start = configs1[count - 1].get_scaled_positions()
        scaled_pos1_end = configs1[count].get_scaled_positions()
        scaled_pos2_start = configs2[j - 1].get_scaled_positions()
        scaled_pos2_end = configs2[j].get_scaled_positions()

        # Корректировка скалированных позиций с учетом PBC
        delta_pos1 = scaled_pos1_end - scaled_pos1_start
        delta_pos2 = scaled_pos2_end - scaled_pos2_start
        delta_pos1 -= np.round(delta_pos1)  # Учитываем PBC
        delta_pos2 -= np.round(delta_pos2)  # Учитываем PBC

        # Создание интерполированных конфигураций с учетом PBC
        interpolated_1 = [configs1[count - 1]] + [configs1[count - 1].copy() for _ in range(10)] + [configs1[count]]
        interpolated_2 = [configs2[j - 1]] + [configs2[j - 1].copy() for _ in range(10)] + [configs2[j]]

        # Интерполяция в скалированных координатах
        for k in range(1, 11):
            alpha = k / 11.0
            new_pos1 = scaled_pos1_start + alpha * delta_pos1
            new_pos2 = scaled_pos2_start + alpha * delta_pos2
            interpolated_1[k].set_scaled_positions(new_pos1)
            interpolated_2[k].set_scaled_positions(new_pos2)

            # Обернуть интерполированные конфигурации с учетом PBC
            interpolated_1[k].wrap()
            interpolated_2[k].wrap()

    else:
        # Условие не выполняется, оставляем кислород (configs2) на месте
        interpolated_1 = [configs1[count - 1]] + [configs1[count - 1].copy() for _ in range(10)] + [configs1[count]]
        
        # Интерполяция для железа
        scaled_pos1_start = configs1[count - 1].get_scaled_positions()
        scaled_pos1_end = configs1[count].get_scaled_positions()
        delta_pos1 = scaled_pos1_end - scaled_pos1_start
        delta_pos1 -= np.round(delta_pos1)  # Учитываем PBC

        for k in range(1, 11):
            alpha = k / 11.0
            new_pos1 = scaled_pos1_start + alpha * delta_pos1
            interpolated_1[k].set_scaled_positions(new_pos1)
            interpolated_1[k].wrap()

        # Оставляем кислород на месте
        interpolated_2 = [configs2[j - 1]] * (10 + 2)  # Копируем неизмененные позиции кислорода

    # Запись интерполированных конфигураций
    for k in range(len(interpolated_1)):
        file.write(f'Direct configuration=     {i:.6f}\n')
        file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_1[k].get_positions()))
        file.write('\n')
        file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_2[k].get_positions()))
        file.write('\n')
        i += 1

    # Запись последней конфигурации
    file.write(f'Direct configuration=     {i:.6f}\n')
    file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs1[count].get_positions()))
    file.write('\n')
    if j < len(configs2):
        file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs2[j-1].get_positions()))
        file.write('\n')

    i += 1
    count += 1

file.close()


2.04

In [110]:
time_steps1[57]

59.766

In [99]:
time_steps2

array([  2.9548,   5.8887,   8.8019,  11.715 ,  14.6282,  17.5414,
        20.4142,  23.3273,  26.2405,  29.1953,  32.1292,  35.0424,
        37.9762,  40.8691,  43.7823,  46.6954,  49.6086,  52.5218,
        55.4557,  58.3895,  61.3443,  64.3204,  67.3399,  70.3376,
        73.3352,  76.29  ,  79.2448,  82.1996,  85.1544,  88.1305,
        91.1281,  94.1476,  97.1671, 100.2089, 103.2284, 106.226 ,
       109.2021, 112.1359, 115.0287, 117.9015, 120.7943, 123.6871,
       126.6003, 129.5341, 132.4473, 135.3402, 138.2129, 141.1057,
       143.9784, 146.8713, 149.7641, 152.6773, 155.5701, 158.4428,
       161.3356, 164.2285, 167.1416, 170.0548, 172.968 , 175.8812,
       178.7944, 181.7283, 184.6831, 187.6379, 190.5927, 193.5903,
       196.588 , 199.5856, 202.5832, 205.5593, 208.5354, 211.533 ,
       214.509 , 217.4639, 220.3977, 223.3525, 226.3286, 229.3262,
       232.281 , 235.2358, 238.149 , 241.1038, 244.0586, 247.0347,
       250.0323, 252.9871, 255.942 , 258.8758, 261.8306, 264.7

In [128]:
import numpy as np
from ase.io.vasp import read_vasp_xdatcar


# Загрузка временных шагов и конфигураций
time_steps1 = np.concatenate(([0], np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1]))
time_steps2 = np.concatenate(([0], np.loadtxt("out_O.txt", usecols=range(6, 10)).T[1]))
configs1 = read_vasp_xdatcar('XDATCAR_Fe', index=slice(None))
configs2 = read_vasp_xdatcar('XDATCAR_O', index=slice(None))

# Установка периодических граничных условий для всех конфигураций
for config in configs1 + configs2:
    config.set_pbc(True)

# Открываем файл для записи
with open('XDATCAR', 'w') as file:
    file.write('Comment\n')
    file.write('1.0\n')
    formatted_matrix = "\n".join("{:.6f} {:.6f} {:.6f}".format(*row) for row in configs1[0].get_cell())
    file.write(formatted_matrix + '\n')

    # Запись информации о типах и количестве атомов
    file.write(f"{configs1[0].get_chemical_symbols()[0]} {configs2[0].get_chemical_symbols()[0]}\n")
    file.write(f"{configs1[0].get_number_of_atoms()} {configs2[0].get_number_of_atoms()}\n")

    i, j, count = 1, 1, 1

    # Основной цикл по временным шагам
    while count < len(time_steps1) and j < len(time_steps2):
        if time_steps1[count] > time_steps2[j]:
            # Временные шаги совпадают, выполняем интерполяцию для обеих конфигураций

            # Получение скалированных позиций для интерполяции
            scaled_pos1_start = configs1[count - 1].get_scaled_positions()
            scaled_pos1_end = configs1[count].get_scaled_positions()
            scaled_pos2_start = configs2[j - 1].get_scaled_positions()
            scaled_pos2_end = configs2[j].get_scaled_positions()

            # Корректировка скалированных позиций с учетом PBC
            delta_pos1 = scaled_pos1_end - scaled_pos1_start
            delta_pos2 = scaled_pos2_end - scaled_pos2_start
            delta_pos1 -= np.round(delta_pos1)  # Учитываем PBC
            delta_pos2 -= np.round(delta_pos2)  # Учитываем PBC

            # Создание интерполированных конфигураций с учетом PBC
            interpolated_1 = [configs1[count - 1]] + [configs1[count - 1].copy() for _ in range(10)] + [configs1[count]]
            interpolated_2 = [configs2[j - 1]] + [configs2[j - 1].copy() for _ in range(10)] + [configs2[j]]

            # Интерполяция в скалированных координатах
            for k in range(1, 11):
                alpha = k / 11.0
                new_pos1 = scaled_pos1_start + alpha * delta_pos1
                new_pos2 = scaled_pos2_start + alpha * delta_pos2
                interpolated_1[k].set_scaled_positions(new_pos1)
                interpolated_2[k].set_scaled_positions(new_pos2)

                # Обернуть интерполированные конфигурации с учетом PBC
                interpolated_1[k].wrap()
                interpolated_2[k].wrap()
                
            j += 1

        else:
            # Условие не выполняется, оставляем кислород (configs2) на месте
            interpolated_1 = [configs1[count - 1]] + [configs1[count - 1].copy() for _ in range(10)] + [configs1[count]]
            
            # Интерполяция для железа
            scaled_pos1_start = configs1[count - 1].get_scaled_positions()
            scaled_pos1_end = configs1[count].get_scaled_positions()
            delta_pos1 = scaled_pos1_end - scaled_pos1_start
            delta_pos1 -= np.round(delta_pos1)  # Учитываем PBC

            for k in range(1, 11):
                alpha = k / 11.0
                new_pos1 = scaled_pos1_start + alpha * delta_pos1
                interpolated_1[k].set_scaled_positions(new_pos1)
                interpolated_1[k].wrap()

            # Оставляем кислород на месте
            interpolated_2 = [configs2[j - 1]] * (10 + 2)  # Копируем неизмененные позиции кислорода

        # Запись интерполированных конфигураций
        for k in range(len(interpolated_1)):
            file.write(f'Direct configuration=     {i:.6f}\n')
            file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_1[k].get_scaled_positions()))
            file.write('\n')
            file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in interpolated_2[k].get_scaled_positions()))
            file.write('\n')
            i += 1

        # # Запись последней конфигурации
        # file.write(f'Direct configuration=     {i:.6f}\n')
        # file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs1[count].get_positions()))
        # file.write('\n')
        # if j < len(configs2):
        #     file.write("\n".join("{:.10f} {:.10f} {:.10f}".format(*row) for row in configs2[j].get_positions()))
        #     file.write('\n')

        i += 1
        count += 1


/Users/oq7884/opt/anaconda3/lib/python3.9/site-packages/ase/atoms.py:966: VisibleDeprecationWarning: Use get_global_number_of_atoms() instead
  warnings.warn('Use get_global_number_of_atoms() instead',


In [121]:
np.concatenate([0], np.loadtxt("out_Fe.txt", usecols=range(6, 10)).T[1])

TypeError: only integer scalar arrays can be converted to a scalar index

In [124]:
time_steps1

array([  0.    ,   1.041 ,   2.0861,   3.1312,   4.1763,   5.2215,
         6.2624,   7.3034,   8.3527,   9.3979,  10.4347,  11.4798,
        12.5291,  13.5701,  14.611 ,  15.6479,  16.6847,  17.7256,
        18.7624,  19.7951,  20.8197,  21.8402,  22.8648,  23.8853,
        24.9019,  25.9184,  26.9271,  27.9397,  28.9562,  29.9728,
        30.9933,  32.0059,  33.0146,  34.0194,  35.0281,  36.0407,
        37.0572,  38.0778,  39.1023,  40.1269,  41.1514,  42.1841,
        43.2168,  44.2374,  45.2619,  46.2825,  47.307 ,  48.3438,
        49.3974,  50.4468,  51.4877,  52.5163,  53.5409,  54.5654,
        55.6022,  56.6432,  57.6841,  58.7251,  59.766 ,  60.8029,
        61.8522,  62.9015,  63.9681,  65.0346,  66.0968,  67.1812,
        68.27  ,  69.3635,  70.4615,  71.5642,  72.6623,  73.7557,
        74.84  ,  75.9154,  76.9908,  78.0706,  79.164 ,  80.2621,
        81.3601,  82.4628,  83.5702,  84.6823,  85.785 ,  86.8924,
        87.9905,  89.0979,  90.2053,  91.3127,  92.4107,  93.5

In [127]:
configs2[1].get_positions()

array([[ 2.15147161,  2.15147161,  2.15147161],
       [10.75735807,  2.15147161,  2.15147161],
       [ 2.15147161, 10.75735807,  2.15147161],
       [10.75735807, 10.75735807,  2.15147161],
       [ 2.15147161,  2.15147161, 10.75735807],
       [10.75735807,  2.15147161, 10.75735807],
       [ 2.15147161, 10.75735807, 10.75735807],
       [10.75735807, 10.75735807, 10.75735807],
       [ 6.45441484,  6.45441484,  6.45441484],
       [15.0603013 ,  6.45441484,  6.45441484],
       [15.0603013 , 15.0603013 ,  6.45441484],
       [ 6.45441484,  6.45441484, 15.0603013 ],
       [15.0603013 ,  6.45441484, 15.0603013 ],
       [ 6.45441484, 15.0603013 , 15.0603013 ],
       [15.0603013 , 15.0603013 , 15.0603013 ],
       [ 6.45441484,  6.45441484,  2.15147161],
       [15.0603013 ,  6.45441484,  2.15147161],
       [15.0603013 , 15.0603013 ,  2.15147161],
       [ 6.45441484,  6.45441484, 10.75735807],
       [15.0603013 ,  6.45441484, 10.75735807],
       [ 6.45441484, 15.0603013 , 10.757